<a href="https://colab.research.google.com/github/nmemranhussain/DiaVigil/blob/main/notebooks%20/%2005_diavigil_agentic_triage_and_faithfulness_critic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Environment, Dependencies & Artifact Deserialization
Python**

In [5]:
# Package Installations
!pip install -q catboost shap pydantic
print("Dependencies installed successfully: catboost, shap, pydantic.")

Dependencies installed successfully: catboost, shap, pydantic.


In [6]:
from google.colab import drive
import os
import joblib
import pandas as pd
import numpy as np
import re
from sklearn.base import BaseEstimator, TransformerMixin

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DiaVigil'
DATA_PROCESSED = os.path.join(BASE_DIR, 'data/processed')

# Install explainability and typing dependencies
!pip install -q shap pydantic
import shap
from pydantic import BaseModel, Field
from typing import List, Literal

# Re-declare custom transformer classes for pipeline deserialization
ELIXHAUSER_ICD9_RULES = {
    'elix_congestive_heart_failure': r'^(39891|40201|40211|40291|40401|40403|40411|40413|40491|40493|428)',
    'elix_cardiac_arrhythmias': r'^(42610|42611|42613|4262|4263|4264|42651|42652|42653|4266|4267|4268|4270|4272|4273|4276|4278|4279|7850|V450|V533)',
    'elix_valvular_disease': r'^(0932|394|395|396|397|424|7463|7464|7465|7466|V422|V433)',
    'elix_pulmonary_circulation': r'^(416|4179)',
    'elix_peripheral_vascular': r'^(440|441|442|443|444|4471|5571|5579|V434)',
    'elix_hypertension_uncomplicated': r'^(401)',
    'elix_hypertension_complicated': r'^(402|403|404|405)',
    'elix_diabetes_complicated': r'^(250[4-9])',
    'elix_renal_failure': r'^(585|586|V420|V451|V56)',
    'elix_fluid_electrolyte': r'^(276)',
    'elix_chronic_pulmonary': r'^(490|491|492|493|494|495|496|500|501|502|503|504|505)',
    'elix_coagulopathy': r'^(286|2871|2873|2874|2875)'
}

def extract_elixhauser_flags(df: pd.DataFrame) -> pd.DataFrame:
    diag_cols = ['diag_1', 'diag_2', 'diag_3']
    comorb_features = pd.DataFrame(index=df.index)
    cleaned_diags = df[diag_cols].fillna('').astype(str).apply(
        lambda col: col.str.strip().str.replace('.', '', regex=False)
    )
    for flag_name, pattern in ELIXHAUSER_ICD9_RULES.items():
        compiled_regex = re.compile(pattern)
        matches = cleaned_diags.apply(
            lambda row: any(bool(compiled_regex.match(val)) for val in row if val),
            axis=1
        )
        comorb_features[flag_name] = matches.astype(int)
    return comorb_features

class ClinicalDomainFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.medication_columns = [
            'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
            'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
            'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
            'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
            'metformin-pioglitazone'
        ]
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_out = X.copy()
        comorb_df = extract_elixhauser_flags(X_out)
        for col in comorb_df.columns:
            X_out[col] = comorb_df[col].values
        X_out['total_elixhauser_burden'] = comorb_df.sum(axis=1).values
        X_out['healthcare_utilization_index'] = (
            X_out['number_outpatient'].astype(float) +
            (2.0 * X_out['number_emergency'].astype(float)) +
            (3.0 * X_out['number_inpatient'].astype(float))
        )
        active_meds = X_out[self.medication_columns].apply(lambda row: (row != 'No').sum(), axis=1)
        med_titrations = X_out[self.medication_columns].apply(lambda row: row.isin(['Up', 'Down']).sum(), axis=1)
        X_out['active_diabetes_med_count'] = active_meds
        X_out['dosage_titration_count'] = med_titrations
        X_out['on_insulin'] = (X_out['insulin'] != 'No').astype(int)
        X_out['encounter_severity_index'] = (
            X_out['time_in_hospital'].astype(float) *
            (X_out['num_medications'].astype(float) / (X_out['num_lab_procedures'].astype(float) + 1.0))
        )
        X_out['glycemic_status_code'] = X_out['A1Cresult'].map({
            'None': 0, 'Norm': 1, '>7': 2, '>8': 2
        }).fillna(0).astype(int)
        return X_out

# Load pipeline, champion model, calibrator, and audit test set
artifacts = joblib.load(os.path.join(DATA_PROCESSED, 'model_calibration_artifacts.joblib'))
test_df = pd.read_parquet(os.path.join(DATA_PROCESSED, 'audit_test_cohort.parquet'))

pipeline = artifacts['fitted_pipeline']
champion_model = artifacts['champion_model']
calibrator = artifacts['calibrator']
optimal_threshold = artifacts['optimal_threshold']

print(f"Setup complete. Model ready for explainability and agent synthesis at tau* = {optimal_threshold:.4f}.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete. Model ready for explainability and agent synthesis at tau* = 0.1620.


**Cell 2: TreeSHAP Attribution Engine**

Initialize TreeSHAP to calculate feature attributions on patient records.

In [7]:
# Transform feature space to retrieve post-encoding feature names
X_sample = test_df.drop(columns=['target', 'encounter_id', 'patient_nbr']).head(100)
transformed_sample = pipeline.transform(X_sample)

# Extract column names from ColumnTransformer
preprocessor = pipeline.named_steps['preprocessor']
feature_names = []
for name, trans, cols in preprocessor.transformers_:
    if name == 'num':
        feature_names.extend(cols)
    elif name == 'comorb':
        feature_names.extend(cols)
    elif name == 'cat':
        ohe_cols = list(trans.named_steps['ohe'].get_feature_names_out(cols))
        feature_names.extend(ohe_cols)

print(f"Total encoded features: {len(feature_names)}")

# Initialize TreeSHAP Explainer on base CatBoost estimator
tree_explainer = shap.TreeExplainer(champion_model)

def extract_patient_shap(patient_row: pd.DataFrame):
    """Computes calibrated probability and top signed SHAP drivers for a single patient."""
    patient_clean = patient_row.drop(columns=['target', 'encounter_id', 'patient_nbr'], errors='ignore')
    patient_trans = pipeline.transform(patient_clean)

    # 1. Calibrated probability
    cal_prob = float(calibrator.predict_proba(patient_trans)[0, 1])
    is_high_risk = bool(cal_prob >= optimal_threshold)

    # 2. Local SHAP values
    shap_vals = tree_explainer.shap_values(patient_trans)[0]

    sorted_idx = np.argsort(shap_vals)
    top_risk_idx = [i for i in sorted_idx[::-1] if shap_vals[i] > 0][:3]
    top_protective_idx = [i for i in sorted_idx if shap_vals[i] < 0][:2]

    risk_drivers = [
        {"feature": feature_names[i], "attribution": round(float(shap_vals[i]), 4)}
        for i in top_risk_idx
    ]
    protective_drivers = [
        {"feature": feature_names[i], "attribution": round(float(shap_vals[i]), 4)}
        for i in top_protective_idx
    ]

    return {
        'encounter_id': int(patient_row['encounter_id'].values[0]),
        'patient_nbr': int(patient_row['patient_nbr'].values[0]),
        'calibrated_risk': round(cal_prob, 4),
        'is_high_risk': is_high_risk,
        'risk_drivers': risk_drivers,
        'protective_drivers': protective_drivers
    }

# Test extraction on first test record
sample_shap = extract_patient_shap(test_df.iloc[[0]])
print("SHAP extraction verified for encounter:", sample_shap['encounter_id'])
print(sample_shap)

Total encoded features: 98
SHAP extraction verified for encounter: 55842
{'encounter_id': 55842, 'patient_nbr': 84259809, 'calibrated_risk': 0.0718, 'is_high_risk': False, 'risk_drivers': [{'feature': 'num_medications', 'attribution': 0.0858}, {'feature': 'num_lab_procedures', 'attribution': 0.0785}, {'feature': 'number_diagnoses', 'attribution': 0.0336}], 'protective_drivers': [{'feature': 'number_inpatient', 'attribution': -0.1912}, {'feature': 'discharge_disposition_id_1', 'attribution': -0.1614}]}


**Cell 3: Strictly Typed Schema Contracts (Pydantic)**

Enforce structured input/output contracts for agent communication.

In [8]:
class FeatureImpact(BaseModel):
    feature: str
    attribution: float

class PatientTriageBrief(BaseModel):
    encounter_id: int
    patient_nbr: int
    readmission_risk: float = Field(ge=0.0, le=1.0)
    risk_tier: Literal["Standard Care", "High Risk - Enhanced Follow-up"]
    primary_risk_drivers: List[FeatureImpact]
    protective_factors: List[FeatureImpact]
    synthesized_care_directives: List[str]
    critic_audit_passed: bool = False
    critic_audit_notes: str = ""

print("Pydantic schema definitions compiled.")

Pydantic schema definitions compiled.


**Cell 4: Agent Orchestration Engine with Deterministic Critic Gate**

Build the end-to-end multi-agent pipeline:

Agent 1 (Clinical Retriever): Safely pulls encounter parameters.

Agent 2 (Inference Core): Computes calibrated probability and local TreeSHAP drivers.

Agent 3 (Synthesis Engine): Synthesizes actionable care directives.

Critic Gate (Grounding Verifier): Verifies that the synthesis directly cites the patient's actual top SHAP drivers, catching and rejecting hallucinated clinical factors.

In [9]:
class MultiAgentClinicalBI:
    def __init__(self, cohort_df):
        self.cohort_df = cohort_df

    def agent_1_retriever(self, encounter_id: int) -> pd.DataFrame:
        """Parameterized encounter retrieval without SQL string interpolation."""
        record = self.cohort_df[self.cohort_df['encounter_id'] == encounter_id]
        if record.empty:
            raise ValueError(f"Encounter ID {encounter_id} not found.")
        return record.iloc[[0]]

    def agent_2_inference(self, patient_row: pd.DataFrame) -> dict:
        """ML inference and TreeSHAP feature attribution."""
        return extract_patient_shap(patient_row)

    def agent_3_synthesizer(self, inference_result: dict, inject_hallucination: bool = False) -> PatientTriageBrief:
        """Drafts clinical care directives. Includes an injection flag for critic validation."""
        risk_drivers = inference_result['risk_drivers']
        prot_drivers = inference_result['protective_drivers']

        directives = []
        if inference_result['is_high_risk']:
            tier = "High Risk - Enhanced Follow-up"
            if inject_hallucination:
                # Deliberate hallucination: cites unsupported factors
                directives.append("Alert: Patient exhibits severe hypertension crisis and elevated potassium.")
                directives.append("Schedule emergency cardiology follow-up within 48 hours.")
            else:
                for driver in risk_drivers:
                    fname = driver['feature'].replace('_', ' ')
                    directives.append(f"Targeted clinical action: Follow up on {fname} prior to 30-day window.")
        else:
            tier = "Standard Care"
            directives.append("Patient within calibrated baseline risk. Provide standard diabetes discharge instructions.")

        return PatientTriageBrief(
            encounter_id=inference_result['encounter_id'],
            patient_nbr=inference_result['patient_nbr'],
            readmission_risk=inference_result['calibrated_risk'],
            risk_tier=tier,
            primary_risk_drivers=[FeatureImpact(**d) for d in risk_drivers],
            protective_factors=[FeatureImpact(**d) for d in prot_drivers],
            synthesized_care_directives=directives,
            critic_audit_passed=False,
            critic_audit_notes="Pending verification"
        )

    def critic_gate(self, brief: PatientTriageBrief) -> PatientTriageBrief:
        """Audits synthesized directives against verified TreeSHAP features."""
        if brief.risk_tier == "Standard Care":
            brief.critic_audit_passed = True
            brief.critic_audit_notes = "Verified: Standard care protocol matches low-risk score."
            return brief

        # Extract features that drove model risk
        top_driver_names = [d.feature.lower() for d in brief.primary_risk_drivers]
        synthesis_text = " ".join(brief.synthesized_care_directives).lower()

        # Check for citation of genuine SHAP drivers
        grounded_count = sum(
            1 for feat in top_driver_names
            if any(term in synthesis_text for term in feat.split('_') if len(term) > 3)
        )

        if grounded_count == 0:
            brief.critic_audit_passed = False
            brief.critic_audit_notes = (
                "AUDIT FAILED (HALLUCINATION DETECTED): Directives cite clinical factors "
                "unsupported by local TreeSHAP attributions. Regeneration required."
            )
        else:
            brief.critic_audit_passed = True
            brief.critic_audit_notes = (
                f"AUDIT PASSED: Care directives ground directly into verified risk drivers ({grounded_count} verified)."
            )
        return brief

    def run_pipeline(self, encounter_id: int, simulate_hallucination: bool = False) -> PatientTriageBrief:
        record = self.agent_1_retriever(encounter_id)
        inf_data = self.agent_2_inference(record)
        draft_brief = self.agent_3_synthesizer(inf_data, inject_hallucination=simulate_hallucination)
        final_brief = self.critic_gate(draft_brief)
        return final_brief

bi_engine = MultiAgentClinicalBI(test_df)
print("Multi-agent clinical engine and critic gate initialized.")

Multi-agent clinical engine and critic gate initialized.


**Cell 5: Deterministic Evaluation & Critic Audit Demonstration**

Test the engine across high-risk encounters to prove that the Critic Gate catches ungrounded syntheses while validating verified outputs.

In [11]:
import os
import joblib
import numpy as np

# 1. Retrieve test predictions from Phase 4 audit results or compute directly
audit_results_path = os.path.join(DATA_PROCESSED, 'fairness_audit_results.joblib')

if os.path.exists(audit_results_path):
    audit_data = joblib.load(audit_results_path)
    test_preds = audit_data['test_preds']
    print("Loaded test_preds successfully from Phase 4 audit artifacts.")
else:
    print("Computing out-of-sample predictions directly from pipeline...")
    X_test_clean = test_df.drop(columns=['target', 'encounter_id', 'patient_nbr'], errors='ignore')
    X_test_trans = pipeline.transform(X_test_clean)
    test_cal_probs = calibrator.predict_proba(X_test_trans)[:, 1]
    test_preds = (test_cal_probs >= optimal_threshold).astype(int)

# 2. Select a confirmed high-risk encounter
high_risk_indices = np.where(test_preds == 1)[0]
if len(high_risk_indices) == 0:
    # Fallback to patient with highest calibrated risk if none exceed tau*
    test_encounter_id = int(test_df.iloc[0]['encounter_id'])
else:
    test_encounter_id = int(test_df.iloc[high_risk_indices[0]]['encounter_id'])

print(f"Selected High-Risk Encounter ID for Audit: {test_encounter_id}")

# 3. Scenario 1: Grounded Synthesis (Expected Pass)
print("\n--- SCENARIO 1: GROUNDED SYNTHESIS (EXPECTED PASS) ---")
passed_brief = bi_engine.run_pipeline(test_encounter_id, simulate_hallucination=False)
print(f"Encounter ID:        {passed_brief.encounter_id}")
print(f"Calibrated Risk:     {passed_brief.readmission_risk * 100:.2f}% | Tier: {passed_brief.risk_tier}")
print(f"Audit Status:        Passed = {passed_brief.critic_audit_passed}")
print(f"Critic Notes:        {passed_brief.critic_audit_notes}")
print("Directives:", passed_brief.synthesized_care_directives)

# 4. Scenario 2: Hallucinated Synthesis (Expected Detection & Rejection)
print("\n--- SCENARIO 2: HALLUCINATED SYNTHESIS (EXPECTED DETECTION & REJECTION) ---")
rejected_brief = bi_engine.run_pipeline(test_encounter_id, simulate_hallucination=True)
print(f"Audit Status:        Passed = {rejected_brief.critic_audit_passed}")
print(f"Critic Notes:        {rejected_brief.critic_audit_notes}")
print("Rejected Directives:", rejected_brief.synthesized_care_directives)

# 5. Serialize demonstration artifacts (compatible with Pydantic v1 and v2)
def export_schema(model_obj):
    return model_obj.model_dump() if hasattr(model_obj, 'model_dump') else model_obj.dict()

agent_artifacts = {
    'sample_passed_brief': export_schema(passed_brief),
    'sample_rejected_brief': export_schema(rejected_brief)
}

agent_export_path = os.path.join(DATA_PROCESSED, 'agent_audit_demonstration.joblib')
joblib.dump(agent_artifacts, agent_export_path)
print(f"\nAgent validation state saved to: {agent_export_path}")

Loaded test_preds successfully from Phase 4 audit artifacts.
Selected High-Risk Encounter ID for Audit: 1157574

--- SCENARIO 1: GROUNDED SYNTHESIS (EXPECTED PASS) ---
Encounter ID:        1157574
Calibrated Risk:     17.79% | Tier: High Risk - Enhanced Follow-up
Audit Status:        Passed = True
Critic Notes:        AUDIT PASSED: Care directives ground directly into verified risk drivers (3 verified).
Directives: ['Targeted clinical action: Follow up on elix diabetes complicated prior to 30-day window.', 'Targeted clinical action: Follow up on total elixhauser burden prior to 30-day window.', 'Targeted clinical action: Follow up on number inpatient prior to 30-day window.']

--- SCENARIO 2: HALLUCINATED SYNTHESIS (EXPECTED DETECTION & REJECTION) ---
Audit Status:        Passed = False
Critic Notes:        AUDIT FAILED (HALLUCINATION DETECTED): Directives cite clinical factors unsupported by local TreeSHAP attributions. Regeneration required.
Rejected Directives: ['Alert: Patient exhi